# 05 · Validation Plan — SPPS + protease stability + permeability + controls

**Standard slot:** *validation plan.* **For Project 09 this means:** turn the top candidates into a
**costed, controlled, peptide-appropriate** wet-lab plan. Peptides are **NOT** validated like E. coli
mini-binders — they are made by **solid-phase peptide synthesis (SPPS)** and need **protease-stability**
and **permeability** assays, plus a binding assay and the mandatory controls (positive: a known
p53-peptide; **scrambled-sequence** negative; unrelated negative). Includes the **D-amino-acid /
stapling** stretch and the **Boltz-2 affinity** ranking scaffold (D4/D5).

A design that passes every filter is a **hypothesis** — synthesis + binding/stability/permeability
assays are what test it, and predicted affinity is unreliable. Needs `results/top_candidates.csv`
(notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the peptide validation plan

Generate a plan card from the top candidates: synthesis (SPPS), assays (binding + **stability** +
**permeability**), controls, timeline, costed reagents. Fill the `<...>` from your own numbers; this is
the deliverable other people will actually read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_mod = top.groupby("modality").size().to_dict() if n_top else {}

plan = f"""# MDM2 Peptide / Macrocycle Validation Plan (Project 09 — by <your name>, <date>)

## Candidates
Top {n_top} candidates carried forward ({by_mod}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. `pae_interaction` is confidence, not affinity;
the Boltz-2 affinity score is a RELATIVE RANK, not a K_D. Predicted affinity for short peptides is unreliable.

## Synthesis strategy (peptides are NOT made in E. coli)
- Linear peptides: solid-phase peptide synthesis (SPPS, Fmoc); HPLC purify; confirm by mass spec.
- Macrocycles: SPPS + cyclization (head-to-tail lactam / side-chain / disulfide, or a hydrocarbon staple).
  D-amino acids / N-methylation introduced here (raises synthesis complexity — see the stretch below).

## Assays (binding -> stability -> permeability -> function)
1. Binding: SPR/BLI vs immobilized MDM2, OR a fluorescence-polarization DISPLACEMENT assay of a labeled
   p53 peptide (does the design displace p53 from MDM2?). Test a dilution series.
2. Protease stability (the key reason to cyclize): serum / trypsin / chymotrypsin half-life, linear vs cyclic.
3. Permeability (the key reason a macrocycle could be oral/cell-penetrant): PAMPA and/or Caco-2.
4. Function (the point): cell-based p53-pathway reactivation (e.g. p53-reporter / p21 induction) in MDM2-amplified cells.

## Controls (MANDATORY)
- Positive: a KNOWN p53-mimetic / stapled peptide (e.g. the ATSP-7041 lineage) -> assay + MDM2 reagent are active.
- Negative (scrambled-sequence): YOUR OWN top design with its sequence scrambled -> must LOSE binding (cleanest specificity control).
- Negative (unrelated): an unrelated peptide of similar length -> should not bind MDM2.

## Realistic expectations
De novo peptide/macrocycle hit rates are MODEST and chemistry-dependent; predicted affinity is
UNRELIABLE for short peptides. Expect to synthesize many to find a few real, stable, permeable binders.
Report the experimental hit rate honestly. Do NOT imply a working peptide or fabricate a K_D.

## Timeline + costed reagents (fill in)
- SPPS synthesis ({n_top} peptides + scrambled-sequence negatives; macrocyclization adds steps): ${'{'}'<...>'{'}'}, <...> weeks.
- MDM2 reagent + labeled p53 peptide (FP) or SPR/BLI chips + positive-control peptide: $<...>.
- Protease-stability + PAMPA/Caco-2 assays: $<...>.
- Personnel / instrument time: <...> weeks.

## Responsible research
Competitive p53-mimetic peptides to a human oncology PPI (MDM2) to restore p53 function (in scope).
Synthesis via a biosecurity-screening provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

## 2 · Build the scrambled-sequence negative controls

The single cleanest specificity control: take each top design and **scramble its sequence** — it should
**lose** binding. Synthesizing these alongside the real designs (same SPPS batch) makes the binding
comparison airtight. Here we scaffold the scramble deterministically.

In [ ]:
import random
import peptide_tools as pt   # pt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_sequence(seq, seed=0):
    """Deterministically shuffle the whole peptide as a NEGATIVE-CONTROL stand-in.
    A scrambled p53-mimetic should lose the Phe/Trp/Leu register that engages the MDM2 cleft."""
    rng = random.Random(seed)
    s = list(seq)
    rng.shuffle(s)
    return "".join(s)

negs = []
if n_top and "sequence" in top.columns:
    # sequences live in the pool CSVs; join them onto the top candidates.
    pools = pd.concat([pd.read_csv("results/linear_designs.csv"),
                       pd.read_csv("results/macrocycle_designs.csv"),
                       pd.read_csv("results/miniprotein_foil_designs.csv")], ignore_index=True)
    seqmap = pools.drop_duplicates("design_id").set_index("design_id")["sequence"]
    for _, r in top.iterrows():
        s = str(seqmap.get(r["design_id"], r.get("sequence", "")))
        if s and s != "nan":
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], modality=r.get("modality"),
                             sequence=scramble_sequence(s, seed=pt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-sequence negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-sequence negatives")
else:
    # The top_candidates table may not carry sequences; pull them from the pools regardless.
    if n_top:
        pools = pd.concat([pd.read_csv("results/linear_designs.csv"),
                           pd.read_csv("results/macrocycle_designs.csv"),
                           pd.read_csv("results/miniprotein_foil_designs.csv")], ignore_index=True)
        seqmap = pools.drop_duplicates("design_id").set_index("design_id")["sequence"]
        for _, r in top.iterrows():
            s = str(seqmap.get(r["design_id"], ""))
            if s and s != "nan":
                negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                                 parent=r["design_id"], modality=r.get("modality"),
                                 sequence=scramble_sequence(s, seed=pt._hashints(r["design_id"]) % 10**6),
                                 role="scrambled-sequence negative control"))
        pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
        print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-sequence negatives")
    else:
        print("Run notebook 04 first to produce results/top_candidates.csv.")

## 3 · (Stretch) D-amino-acid / stapling extension `[stretch]`

Cyclization is not the only stability/permeability lever. **D-amino-acid substitutions**, **N-methylation**,
and **hydrocarbon stapling** further resist proteases and can improve permeability — at a real synthesis
cost. Propose specific modifications for your top macrocycles and reason about the trade-off. This is a
*design proposal*, not a prediction — do not claim a stability/permeability number you have not measured.

In [ ]:
# Scaffold ONLY — a design proposal, not a measurement.
# For each top macrocycle, propose (and justify) modifications, then weigh synthesis cost vs benefit:
#   - D-amino-acid substitution at protease-cleavage-prone positions (e.g. after Arg/Lys);
#   - N-methylation of backbone amides to block proteolysis + aid membrane permeability;
#   - a hydrocarbon staple (i, i+4 / i, i+7) to lock the bound helix-mimetic conformation.
# Report these as HYPOTHESES; protease half-life + PAMPA/Caco-2 in the wet-lab plan TEST them.
print("D-amino-acid / N-methylation / stapling = stretch DESIGN proposals (stability/permeability levers).")
print("Weigh synthesis complexity vs the protease-stability / permeability benefit. Measure, never assume.")

## 4 · (Stretch) Boltz-2 affinity to prioritize synthesis order `[stretch]`

Boltz-2 can predict an affinity *signal* for the top complexes. Use it for **relative ranking +
caveats only** — to decide **which hits to synthesize first** — **never** as evidence of binding and
**never** as a K_D. (You already carried `boltz_affinity_score` through the pipeline; here it only sets
priority.)

In [ ]:
import pandas as pd, os
if os.path.exists("results/top_candidates.csv"):
    top = pd.read_csv("results/top_candidates.csv")
    if "boltz_affinity_score" in top.columns and top["boltz_affinity_score"].notna().any():
        order = top.sort_values("boltz_affinity_score", ascending=False)
        print("Suggested SYNTHESIS ORDER by Boltz-2 RELATIVE rank (NOT a K_D; SYNTHETIC if mock):")
        print(order[["design_id", "modality", "boltz_affinity_score"]].head(10).to_string(index=False))
    else:
        print("No Boltz-2 rank present — populate boltz_affinity_score on Colab (relative rank only).")
print("\nReminder: Boltz-2 affinity = a RELATIVE prioritization signal. NEVER a fabricated K_D (MASTER_BLUEPRINT §9).")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **SPPS** synthesis + binding + **protease-stability** + **permeability** + function, timeline, costed reagents.
- [ ] Controls specified: positive (known p53-peptide / stapled peptide), **scrambled-sequence** negative (`results/negative_controls.csv`), unrelated negative.
- [ ] (Stretch) D-amino-acid / N-methylation / stapling proposals for top macrocycles, with the synthesis-cost trade-off.
- [ ] (Stretch) Boltz-2 affinity used only to set synthesis priority — relative rank, no fabricated K_D.
- [ ] Honest framing: every design is a hypothesis until synthesis + assays; report the experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a peptide/macrocycle capstone that mirrors the binder-family workflow with peptide
chemistry, SPPS validation, and the linear-vs-cyclic + peptide-vs-protein modality study at its core.